# EfficientNet-B0 Experiment 06 — TTA on Best Iter 05 Checkpoint

## Objective
Apply Test-Time Augmentation (TTA) to the saved checkpoint from Iteration 05 (`efficientnet_b0_l1_metadata_best.pth`) without any additional training. Iter 05 achieved Test F2 **0.6875** (AUC 0.9199), sitting 0.0004 short of the 0.6879 benchmark. TTA averages predictions over 8 geometric/photometric augmentations of each image, keeping the patient metadata fixed, to reduce prediction variance and potentially close that gap.

## Changes vs Iter 05

| Component | Iter 05 | Iter 06 (TTA) |
|---|---|---|
| Training | 30 epochs | None (frozen inference) |
| Checkpoint | `efficientnet_b0_l1_metadata_best.pth` | same |
| Inference | Single pass, eval transforms | 8-augment TTA, averaged |
| Metadata | Passed per sample | Passed per sample (unchanged) |

## TTA Transforms (8 augmentations)
1. Identity (resize + normalise)
2. Horizontal flip
3. Vertical flip
4. Horizontal + Vertical flip
5. Rotate 90°
6. Rotate 180°
7. Rotate 270°
8. Mild ColorJitter (brightness/contrast ±0.1)

In [7]:
import sys
from pathlib import Path

import numpy as np
import torch
from torchvision import transforms
from sklearn.metrics import fbeta_score, roc_auc_score, balanced_accuracy_score, classification_report

ROOT = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / 'src').exists())
sys.path.insert(0, str(ROOT))

from src.data.dataset import HAM10000DatasetWithMetadata
from src.models.efficientnet import EfficientNetB0WithMetadata
from src.utils import seed_everything

seed_everything(42)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

Using device: cuda


## Load Datasets (no transform — TTA applies transforms manually)

In [8]:
METADATA_PATH      = str(ROOT / 'data_new/raw/HAM10000_metadata')
TEST_METADATA_PATH = str(ROOT / 'data_new/raw/ISIC2018_Task3_Test_GroundTruth.csv')

# transform=None so TTA can apply different transforms to the raw PIL image
val_dataset = HAM10000DatasetWithMetadata(
    csv_path=str(ROOT / 'data_new/splits/val.csv'),
    image_dir=str(ROOT / 'data_new/images/train'),
    metadata_path=METADATA_PATH,
    transform=None,
)

test_dataset = HAM10000DatasetWithMetadata(
    csv_path=str(ROOT / 'data_new/splits/test.csv'),
    image_dir=str(ROOT / 'data_new/images/test'),
    metadata_path=TEST_METADATA_PATH,
    transform=None,
)

print(f'Val: {len(val_dataset)} | Test: {len(test_dataset)}')

Val: 2024 | Test: 1511


## Load Model from Iter 05 Checkpoint

In [9]:
METADATA_DIM = 17
DROPOUT      = 0.5

model = EfficientNetB0WithMetadata(
    metadata_dim=METADATA_DIM,
    num_classes=1,
    freeze_backbone=True,
    dropout=DROPOUT,
).to(device)

model.load_state_dict(torch.load(
    str(ROOT / 'models/efficientnet_b0_l1_metadata_best.pth'),
    map_location=device,
))
model.eval()
print('Checkpoint loaded: efficientnet_b0_l1_metadata_best.pth')

Checkpoint loaded: efficientnet_b0_l1_metadata_best.pth


## TTA Setup

In [10]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def _base(extra=None):
    ops = [transforms.Resize((224, 224))]
    if extra:
        ops += extra
    ops += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(ops)

tta_transforms = [
    _base(),                                                                      # 1. identity
    _base([transforms.RandomHorizontalFlip(p=1.0)]),                              # 2. H-flip
    _base([transforms.RandomVerticalFlip(p=1.0)]),                                # 3. V-flip
    _base([transforms.RandomHorizontalFlip(p=1.0),
           transforms.RandomVerticalFlip(p=1.0)]),                                # 4. HV-flip
    _base([transforms.RandomRotation(degrees=(90, 90))]),                         # 5. rotate 90
    _base([transforms.RandomRotation(degrees=(180, 180))]),                       # 6. rotate 180
    _base([transforms.RandomRotation(degrees=(270, 270))]),                       # 7. rotate 270
    _base([transforms.ColorJitter(brightness=0.1, contrast=0.1)]),                # 8. color jitter
]


def tta_predict(model, dataset, device, tta_transforms):
    """Run TTA over a dataset, returning (probs, labels) as numpy arrays.
    
    For each sample: apply all TTA transforms to the PIL image, pass each through
    the model with the fixed metadata tensor, then average the sigmoid probabilities.
    """
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for idx in range(len(dataset)):
            pil_img, metadata, label = dataset[idx]  # pil_img is raw PIL (transform=None)
            metadata = metadata.unsqueeze(0).to(device)  # (1, 17)

            preds = []
            for t in tta_transforms:
                tensor = t(pil_img).unsqueeze(0).to(device)  # (1, 3, 224, 224)
                prob = torch.sigmoid(model(tensor, metadata)).item()
                preds.append(prob)

            all_probs.append(np.mean(preds))
            all_labels.append(label)

    return np.array(all_probs), np.array(all_labels)


print(f'TTA: {len(tta_transforms)} augmentations per sample')

TTA: 8 augmentations per sample


## Threshold Tuning on Validation Set

In [11]:
print('Running TTA on validation set...')
val_probs, val_labels = tta_predict(model, val_dataset, device, tta_transforms)

thresholds = np.arange(0.01, 0.90, 0.01)
f2_scores  = [
    fbeta_score(val_labels, (val_probs >= t).astype(int), beta=2, pos_label=1, zero_division=0)
    for t in thresholds
]
best_threshold = thresholds[np.argmax(f2_scores)]
best_val_f2    = max(f2_scores)

print(f'Best threshold: {best_threshold:.2f} | Val F2: {best_val_f2:.4f}')
print(f'(Iter 05 without TTA: threshold=0.62, val F2=0.6872)')

Running TTA on validation set...
Best threshold: 0.54 | Val F2: 0.6887
(Iter 05 without TTA: threshold=0.62, val F2=0.6872)


## Test Set Evaluation

In [12]:
print('Running TTA on test set...')
test_probs, test_labels = tta_predict(model, test_dataset, device, tta_transforms)
all_preds = (test_probs >= best_threshold).astype(int)

auc     = roc_auc_score(test_labels, test_probs)
bal_acc = balanced_accuracy_score(test_labels, all_preds)
f2      = fbeta_score(test_labels, all_preds, beta=2, pos_label=1, zero_division=0)

print(f'Threshold:          {best_threshold:.2f}')
print(f'AUC-ROC:            {auc:.4f}')
print(f'Balanced Accuracy:  {bal_acc:.4f}')
print(f'F2 Score:           {f2:.4f}')
print()
print(classification_report(
    test_labels, all_preds,
    target_names=['Non-Melanoma', 'Melanoma'],
    digits=4,
))
print(f'\n--- Comparison ---')
print(f'Iter 05 (no TTA): Test F2=0.6875 | AUC=0.9199 | Recall=0.8363 | Precision=0.4017')
print(f'Iter 06 (TTA):    Test F2={f2:.4f} | AUC={auc:.4f}')

Running TTA on test set...
Threshold:          0.54
AUC-ROC:            0.9184
Balanced Accuracy:  0.8479
F2 Score:           0.6952

              precision    recall  f1-score   support

Non-Melanoma     0.9820    0.8127    0.8893      1340
    Melanoma     0.3756    0.8830    0.5271       171

    accuracy                         0.8206      1511
   macro avg     0.6788    0.8479    0.7082      1511
weighted avg     0.9133    0.8206    0.8483      1511


--- Comparison ---
Iter 05 (no TTA): Test F2=0.6875 | AUC=0.9199 | Recall=0.8363 | Precision=0.4017
Iter 06 (TTA):    Test F2=0.6952 | AUC=0.9184
